# Building an MLP for weigths recovery in Pott's Model
This file defined an MLP that will try to learn the weights in profile and Pott's model, this consideration is due to a difficulty in finding a good penalization factor during ridge regression, the too many weights create a situation where the best lambda is extremely high, and therefore no choices are made and the predicted weights are too near the mean, therefore zero        

### Import of useful libraries

In [1]:
!pip install -U "jax[cuda12]"
!pip install -U flax

INFO: pip is looking at multiple versions of jax[cuda12] to determine which version is compatible with other requirements. This could take a while.
  Using cached jax-0.11.0-py3-none-any.whl.metadata (13 kB)
  Using cached jax-0.10.2-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.10.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (1.3 kB)
  Using cached jax-0.10.1-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.10.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (1.3 kB)
  Using cached jax-0.10.0-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.10.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (1.3 kB)
INFO: pip is still looking at multiple versions of jax[cuda12] to determine which version is compatible with other requirements. This could take a while.
  Using cached jax-0.9.2-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.9.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (1.3 kB)
  Using cached jax-0.9.1-py3-none-any.whl.metadata (13 kB)
  Using cached ja

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import jax 
import jax.numpy as jnp

from flax import nnx
from functools import partial
from typing import Optional

### First simple MLP

In [12]:
class SimpleMLP(nnx.Module):
    """ 
    Simple MLP
    """
    
    def __init__(self, input_dim:int, hidden_dim:int, output_dim:int, *, rngs: nnx.Rngs):
        self.linear1    = nnx.Linear(input_dim, hidden_dim, rngs=rngs)
        self.dropout    = nnx.Dropout(rate=0.1, rngs=rngs)
        self.batchnorm  = nnx.BatchNorm(hidden_dim, use_running_average=False, rngs=rngs)
        self.linear2    = nnx.Linear(hidden_dim, output_dim, rngs=rngs)
        
    def __call__(self, x:jax.Array, rngs: nnx.Rngs):
        x = nnx.gelu(self.dropout(self.batchnorm(self.linear1(x)), rngs=rngs))
        return self.linear2(x)

model = SimpleMLP(2, 256, 16, rngs=nnx.Rngs(0))

y = model(x=jnp.ones((3,2)), rngs=nnx.Rngs(1))

nnx.display(model)